# Stochastic White Noise Analysis (SWNA) with Memory
### Workshop — Hands-on Session

**Package:** `whitenoise` — implements the SWNA with Memory framework for stochastic analysis of time series data.

---

## The Pipeline

```
Your CSV file
     ↓
  read_csv()          → loads time, values, and column labels
     ↓
  [optional: detrend() / normalize()]   ← preprocessing if needed
     ↓
  analyze()           → computes MSD, fits model, extracts μ
     ↓
  result.summary()    → prints μ, R², regime
  result.plot_all()   → time series + MSD fit + PDF
```

**The memory parameter μ tells you:**
| μ range | Regime |
|---|---|
| μ < 0.95 | Subdiffusive — system resists change |
| 0.95 – 1.05 | Near-Brownian — weak or no memory |
| 1.05 – 2.0 | Superdiffusive — past reinforces future |
| μ > 2.0 | Hyperballistic — strongly persistent |

---
## Step 0 — Install the package

Run this once at the start of every session. It takes about 30 seconds.

In [ ]:
!pip install git+https://github.com/pnayga/whitenoise.git -q
print("Installation complete.")

In [ ]:
import whitenoise as wn
print(f"whitenoise version: {wn.__version__}")

---
## Step 1 — What models are available?

The package implements 16 models from Table 3.1 of Bernido & Carpio-Bernido (2015).
The four fully implemented models are: **cosine**, **exponential**, **sine**, **fbm**, **dna**.

In [ ]:
wn.list_models()

---
## Step 2 — CSV format

Your CSV must follow this exact format:

```
time [months], sunspot_number [count]
1, 12.4
2, 15.1
3, 14.8
```

Rules:
- **Row 1** is always the header
- **Column 1** = time or index (independent variable)
- **Column 2** = your observable (what you measured)
- Header format: `name [unit]` — the unit goes inside square brackets
- Unitless columns: use empty brackets → `flux []`

---
## Step 3 — Demo: Analyzing a sample dataset

We'll use a sample time series (fractional Brownian motion, H ≈ 0.72) to walk through the full pipeline.
This dataset has a known answer — superdiffusive dynamics with H ≈ 0.72 — so you can verify the fit works correctly.

In [ ]:
# Download the sample dataset from GitHub
!wget -q https://raw.githubusercontent.com/pnayga/whitenoise/main/sample_data/sample_series.csv
print("Sample file downloaded: sample_series.csv")

In [ ]:
# Peek at the file to confirm the format
with open("sample_series.csv") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            print("...")
            break

In [ ]:
# Read the CSV
time, values, meta = wn.read_csv("sample_series.csv")

print(f"Variable : {meta['y_name']}")
print(f"Unit     : {meta['y_unit']}")
print(f"N points : {meta['n_points']}")
print(f"Index    : {time[0]:.0f} to {time[-1]:.0f} {meta['x_unit']}")

In [ ]:
# Quick look at the raw time series
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3.5))
plt.plot(time, values, color='#2C3E50', lw=1.0, alpha=0.85)
plt.xlabel(meta['x_label'])
plt.ylabel(meta['y_label'])
plt.title(meta['y_name'])
plt.grid(alpha=0.25, linestyle='--')
plt.tight_layout()
plt.show()

---
## Step 4 — Run the full analysis

In [ ]:
# Analyze with the fbm model (fractional Brownian motion)
# Expected: H ≈ 0.72, regime = superdiffusive
result = wn.analyze("sample_series.csv", model="fbm")

In [ ]:
# Print the summary
result.summary()

In [ ]:
# Publication-quality plots: time series + MSD fit + PDF
fig = wn.plot_diagnostics(result)
plt.show()

In [ ]:
# Access individual results
print(f"H (Hurst exponent)  : {result.fit.params['H']:.4f}   (expected ≈ 0.72)")
print(f"R²                  : {result.fit.r_squared:.4f}")
print(f"Regime              : {result.regime}")

---
## Step 5 — Try a different model

If R² is low, try a different model. The researcher always chooses — the package does not auto-select.

In [ ]:
# Try a different model to see how R² changes
result_cos = wn.analyze("sample_series.csv", model="cosine")

In [ ]:
# Compare R² — fbm should win because the data was generated from fBm
print(f"fbm     R² = {result.fit.r_squared:.4f}   H  = {result.fit.params['H']:.4f}")
print(f"cosine  R² = {result_cos.fit.r_squared:.4f}   mu = {result_cos.fit.params['mu']:.4f}")
print()
print("→ Higher R² = better model choice for this dataset")

---
## Step 6 — Optional: Preprocessing before analysis

Use these if your data has a trend or is not in a comparable scale.

In [ ]:
# Detrend: remove a linear trend before analysis
time, values, meta = wn.read_csv("sample_series.csv")
fluctuations = wn.detrend(values, method='linear')

# Analyze the detrended data
result_detrended = wn.analyze(
    fluctuations,
    model='fbm',
    label='sample_detrended'
)
result_detrended.summary()

---
---
# ✋ Hands-on: Analyze Your Own Data

Now it's your turn. Upload your own CSV file and run the analysis.

**Your CSV must have this format:**
```
time [unit], variable_name [unit]
1, 23.5
2, 24.1
...
```

In [ ]:
# Upload your CSV file
from google.colab import files
uploaded = files.upload()

# Get the filename that was uploaded
my_file = list(uploaded.keys())[0]
print(f"Uploaded: {my_file}")

In [ ]:
# Check the file contents
with open(my_file) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            print("...")
            break

In [ ]:
# Choose your model: 'cosine', 'exponential', 'sine', or 'fbm'
MY_MODEL = 'cosine'   # ← change this

my_result = wn.analyze(my_file, model=MY_MODEL)
my_result.summary()

In [ ]:
# Plot the results
fig = wn.plot_diagnostics(my_result)
plt.show()

In [ ]:
# Save your results to a CSV
wn.export_csv(my_result, "my_results.csv")

# Download the results file
files.download("my_results.csv")

---
## Bonus: Compare multiple datasets

If you have more than one CSV, you can compare them in one call.

In [ ]:
# Upload multiple files
uploaded_multi = files.upload()
file_list = list(uploaded_multi.keys())
print(f"Uploaded {len(file_list)} files: {file_list}")

In [ ]:
# Compare all uploaded files
comparison = wn.compare(file_list, model='cosine')
wn.print_comparison(comparison)

In [ ]:
# Bar chart: μ values across all datasets
fig = wn.publish_comparison(comparison)
plt.show()

---
## Quick Reference

| Function | What it does |
|---|---|
| `wn.analyze(file, model='cosine')` | Full pipeline → AnalysisResult |
| `result.summary()` | Print μ, R², regime |
| `wn.plot_diagnostics(result)` | 4-panel diagnostic figure |
| `wn.detrend(values, method='linear')` | Remove linear trend |
| `wn.normalize(values, method='zscore')` | Z-score normalization |
| `wn.compare([file1, file2], model='cosine')` | Compare multiple datasets |
| `wn.export_csv(result, 'out.csv')` | Save results to CSV |
| `wn.list_models()` | Show all 16 models and status |

**Models:** `cosine` · `exponential` · `sine` · `fbm` · `dna`

**Preprocessing:** `wn.detrend()` · `wn.normalize()` · `wn.smooth()`